## 📹 Video Introduction

Watch this video before starting the module:

<div style="padding:56.25% 0 0 0;position:relative;">
  <iframe src="https://player.vimeo.com/video/YOUR_VIDEO_ID" style="position:absolute;top:0;left:0;width:100%;height:100%;" frameborder="0" allow="autoplay; fullscreen; picture-in-picture" allowfullscreen></iframe>
</div>

---

## 💻 Local Setup

In lessons 3-5 you built your own robot. From this lesson onwards we switch to **Turtlebot3** — a standard ROS2 reference robot that comes pre-configured with a LiDAR sensor and full Nav2 compatibility. In a real project you would adapt your own robot the same way.

This module runs **entirely in your local ROS2 environment** using the container we configured earlier.

Clone the Turtlebot3 packages into a local workspace:

```bash
mkdir -p ~/turtlebot3_ws/src
cd ~/turtlebot3_ws/src
git clone -b jazzy https://github.com/ROBOTIS-GIT/turtlebot3_simulations.git
git clone -b jazzy https://github.com/ROBOTIS-GIT/turtlebot3.git
cd ~/turtlebot3_ws
colcon build --symlink-install
source install/setup.bash
```

Install **slam_toolbox** (if not already present in your ROS 2 installation):

```bash
sudo apt update
sudo apt install ros-jazzy-slam-toolbox
```

Add to `~/.bashrc` so it loads automatically in every terminal:

```bash
echo 'source ~/turtlebot3_ws/install/setup.bash' >> ~/.bashrc
echo 'export TURTLEBOT3_MODEL=waffle' >> ~/.bashrc
source ~/.bashrc
```

Verify:

```bash
echo $TURTLEBOT3_MODEL
```

You should see `waffle`.

## 🧠 What is SLAM?

**SLAM** stands for **S**imultaneous **L**ocalization **A**nd **M**apping.

A robot performing SLAM does two things at the same time:

1. **Localization** — estimate where it is
2. **Mapping** — build a map of the environment

This is difficult because the robot doesn't know the map, and doesn't know its position. Both must be estimated **simultaneously**.

SLAM solves this using laser scans, odometry, and probabilistic estimation. The result is a **2D occupancy grid map**.

![Two-step process: Phase 1 SLAM then Phase 2 Navigate](images/nav2_slide03.jpg)

---

## 🗺️ Occupancy Grid Maps

The environment is divided into small cells. Each cell represents whether the area is occupied.

| Cell Value | Meaning |
|-------------|--------|
| 0 | Free space |
| 100 | Obstacle |
| -1 | Unknown |

As the robot moves and senses obstacles, the map gradually appears.

---

## 🔧 SLAM Architecture in ROS 2

### Required Inputs

- `/scan` → Laser scanner data
- `/odom` → Wheel odometry
- `/tf` → Coordinate frame transforms

### Output

- `/map` → Generated occupancy grid
- `/tf` → `map → odom` transform

The SLAM node continuously updates the map while the robot moves.

---

![SLAM architecture in ROS2](images/nav2_architecture.png)

## 🛠️ Why slam_toolbox?

**slam_toolbox** is the recommended SLAM solution for ROS 2 and the default used by Nav2. It offers:

| Feature | slam_toolbox |
|---|---|
| Lifecycle node support | ✅ Native |
| Map serialization (pause & resume) | ✅ Yes |
| Localization-only mode | ✅ Yes |
| Nav2 integration | ✅ First-class |
| Default in Nav2 Bringup | ✅ Yes |

slam_toolbox runs in **online async** mode by default, which means it processes scans as they arrive without blocking the robot's motion.

---

## 🚀 Running the Full Simulation

Open several terminals.

### Terminal 1 — Start Gazebo with Turtlebot3

```bash
ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py
```

### Terminal 2 — Start SLAM with slam_toolbox

```bash
ros2 launch slam_toolbox online_async_launch.py \
  use_sim_time:=True \
  slam_params_file:=$(ros2 pkg prefix slam_toolbox)/share/slam_toolbox/config/mapper_params_online_async.yaml
```

> **Note:** The `online_async_launch.py` file is included with the `slam_toolbox` package. It starts the SLAM node in online asynchronous mode, which is appropriate for live simulation.

### Terminal 3 — Teleoperate the Robot

```bash
ros2 run turtlebot3_teleop teleop_keyboard
```

Drive the robot around the entire environment. The map will appear gradually in RViz.

---

## ⚙️ slam_toolbox Parameters

The default config file is located at:

```bash
$(ros2 pkg prefix slam_toolbox)/share/slam_toolbox/config/mapper_params_online_async.yaml
```

Key parameters to know:

| Parameter | Default | Meaning |
|---|---|---|
| `resolution` | `0.05` | Map cell size in meters |
| `max_laser_range` | `20.0` | Maximum range of laser scans used |
| `minimum_travel_distance` | `0.5` | Distance to move before adding a new scan |
| `minimum_travel_heading` | `0.5` | Rotation (rad) before adding a new scan |
| `map_update_interval` | `5.0` | How often (s) the map topic is updated |
| `use_sim_time` | `false` | Set to `true` for Gazebo simulation |

You can copy and modify this file to tune the mapper for your environment.

---

## 👁️ Visualizing the Map in RViz

### Terminal 4 — Open RViz

```bash
rviz2
```

In RViz, configure the following:

1. **Fixed Frame** → set to `map`
2. **Add → By topic → /map** → shows the occupancy grid
3. **Add → By topic → /scan** → shows live laser readings
4. **Add → TF** → visualizes coordinate frames

> Drive slowly and cover every room — unexplored zones remain grey.

slam_toolbox also publishes a `/slam_toolbox/scan_visualization` topic that you can add to RViz to see which scans are being matched.

---

![Map building in RViz](images/navigation_stack.png)

## 💾 Saving the Map

### Option A — Using map_saver_cli (standard)

Once the robot has explored the environment, save the generated map:

```bash
ros2 run nav2_map_server map_saver_cli -f ~/my_map
```

This generates two files:

```
my_map.pgm   ← the map image
my_map.yaml  ← metadata: resolution, origin
```

### Option B — Using slam_toolbox's built-in serialization

slam_toolbox can also serialize its internal map representation, allowing you to **resume mapping later** from where you left off:

```bash
ros2 service call /slam_toolbox/save_map slam_toolbox/srv/SaveMap "{name: {data: 'my_map'}}"
```

Or serialize the full solver state (to resume mapping):

```bash
ros2 service call /slam_toolbox/serialize_map slam_toolbox/srv/SerializePoseGraph "{filename: '/home/$USER/my_map'}"
```

> For navigation with Nav2, always use **Option A** — Nav2's map server loads `.pgm` + `.yaml` files.

![Map anatomy: pgm pixel values and yaml metadata](images/nav2_slide12.jpg)

---

## 🧠 Challenge

Open your saved map files:

```bash
evince ~/my_map.pgm
cat ~/my_map.yaml
```

1. What is the **resolution** of your map? What does that number mean in meters per pixel?
2. Are there any **grey areas** left? What caused them?
3. What happens if you send Nav2 a goal through a grey zone?
4. *(Bonus)* Try adjusting `minimum_travel_distance` in the slam_toolbox config to `0.2`. How does this affect mapping quality and CPU usage?

*Write your answers here:*

**1. Resolution:**

**2. Grey areas:**

**3. Nav2 through grey zone:**

**4. Bonus — effect of reducing minimum_travel_distance:**

# Summary

In this module, you learned:

✅ What SLAM is and why robots need it

✅ How robots build occupancy grid maps

✅ How to run slam_toolbox in ROS 2 (online async mode)

✅ Key slam_toolbox parameters and how to tune them

✅ How to visualize the map in RViz

✅ How to save a generated map for later use with Nav2

---

> Your robot can now create a map of its environment. But mapping is only the beginning. The next challenge is navigation: how the robot plans a path and moves autonomously to a goal. In the next lesson we will introduce the ROS2 Navigation Stack (Nav2).